# Experiment 2: Metadata 
This second experiment consists on seeing which is the best technique concerning metadata: 
- No using metadata
- Using metadata and concatenate it to the text embeddings 
- Using metadata with it's separate embeddings and have a score on both embeddings

In [1]:
import os, json
import google.generativeai as genai
import uuid
import fitz  #pip install pymupdf
import json
import tempfile
import requests
import numpy as np
from pathlib import Path
import sys

from embedder import Embedder

C:\Users\lucia\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\lucia\AppData\Local\Temp\ipykernel_29544\2124496564.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:


# we are in: backend/exp2/experiment_2_metadata.ipynb
BASE_DIR = Path.cwd().parents[0]   # backend/
DATA_DIR = BASE_DIR / "data"
PDF_DIR = DATA_DIR / "pdfs"
DB_PATH = DATA_DIR / "database.json"

sys.path.append(str(BASE_DIR))

print("PDF_DIR:", PDF_DIR)
print("PDFs:", len(list(PDF_DIR.glob("*.pdf"))))
print("DB exists:", DB_PATH.exists())

PDF_DIR: c:\Lucía\Lucia\uni\z otras cosas\ERASMUS\VIENA\asignaturas\GenAI\GenAI-PR-2025w\backend\data\pdfs
PDFs: 15
DB exists: True


In [3]:
with open(DB_PATH, "r", encoding="utf-8") as f:
    db = json.load(f)

pdf_names = [e["pdf_name"] for e in db if "pdf_name" in e]
pdf_paths = [PDF_DIR / name for name in pdf_names if (PDF_DIR / name).exists()]

print("PDFs in DB:", len(pdf_names))
print("PDFs found on disk:", len(pdf_paths))
pdf_paths[:3]

PDFs in DB: 15
PDFs found on disk: 15


[WindowsPath('c:/Lucía/Lucia/uni/z otras cosas/ERASMUS/VIENA/asignaturas/GenAI/GenAI-PR-2025w/backend/data/pdfs/1907.02052v1.pdf'),
 WindowsPath('c:/Lucía/Lucia/uni/z otras cosas/ERASMUS/VIENA/asignaturas/GenAI/GenAI-PR-2025w/backend/data/pdfs/1911.00536v3.pdf'),
 WindowsPath('c:/Lucía/Lucia/uni/z otras cosas/ERASMUS/VIENA/asignaturas/GenAI/GenAI-PR-2025w/backend/data/pdfs/1911.02365v1.pdf')]

In [ ]:
#gemini metadata 

#EVERYONE NEEDS THEIR API KEY
#environment variable
#Windows powershell => setx GEMINI_API_KEY "YOUR_API_KEY"
#Linux/MacOs => export GEMINI_API_KEY="YOUR_API_KEY"
os.environ["GEMINI_API_KEY"] = "ATAAT"
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY env var")


genai.configure(api_key=api_key)

#GEMINI_MODEL = "gemini-2.0-flash" 
GEMINI_MODEL = "gemini-2.0-flash-lite"

META_CACHE_PATH = DATA_DIR / "llm_metadata_cache.json"

if META_CACHE_PATH.exists():
    metadata_cache = json.loads(META_CACHE_PATH.read_text(encoding="utf-8"))
else:
    metadata_cache = {}


def extract_llm_metadata(doc_text):
    """
    Returns metadata in JSON from the text of the document
    """

    #client = genai.Client(api_key=api_key)

    # recorta para no pasarle el documento entero (suficiente con inicio/abstract)
    snippet = doc_text[:12000]

    prompt = f"""
        You are extracting bibliographic and topical metadata from a PDF text dump.
        Return ONLY valid JSON (no markdown).

        Schema:
        {{
        "title": string|null,
        "authors": [string],
        "year": int|null,
        "keywords": [string],   // 8-15 items
        "topics": [string],     // 2-5 short tags
        "one_sentence_summary": string|null
        }}

        Rules:
        - If you are unsure, use null or empty lists.
        - Keep keywords/topics concise (1-4 words).
        - Do not hallucinate specific author names if not present.
        - Base everything only on the provided text.

        TEXT:
        {snippet}
        """.strip()

    model = genai.GenerativeModel(GEMINI_MODEL)


    resp = model.generate_content(prompt)
    return json.loads(resp.text.strip())


import time
def get_llm_metadata(pdf_name, doc_text):
    if pdf_name not in metadata_cache:
        time.sleep(4)
        metadata_cache[pdf_name] = extract_llm_metadata(doc_text)
        META_CACHE_PATH.write_text(json.dumps(metadata_cache, indent=2), encoding="utf-8")
    return metadata_cache[pdf_name]

def build_metadata_string(llm_meta):

    if not llm_meta:
        return ""
    parts = []
    if llm_meta.get("title"):
        parts.append(f"Title: {llm_meta['title']}")
    if llm_meta.get("topics"):
        parts.append("Topics: " + ", ".join(llm_meta["topics"]))
    if llm_meta.get("keywords"):
        parts.append("Keywords: " + ", ".join(llm_meta["keywords"]))
    if llm_meta.get("one_sentence_summary"):
        parts.append("Summary: " + llm_meta["one_sentence_summary"])
    return "[METADATA]\n" + "\n".join(parts)

In [ ]:
# data_manager.py

class DataManager:
    """
    Handles everything related to:
    - downloading or receiving PDFs
    - saving them to the database
    - extracting text
    - chunking, embedding & storing database
    """

    def __init__(self, database_file=DB_PATH, pdf_folder=PDF_DIR, metadata_mode = "concat"):
        self.embedder = Embedder()
        self.database_file = database_file
        self.pdf_folder = pdf_folder
        #self.database = self.load_database()
        self.metadata_mode = metadata_mode #"concat" or "dual"

    # ------------------------------
    # DATABASE I/O
    # ------------------------------
    '''    def load_database(self):
            if os.path.exists(self.database_file):
                with open(self.database_file, "r", encoding="utf-8") as f:
                    return json.load(f)
            return []

        def save_database(self):
            with open(self.database_file, "w", encoding="utf-8") as f:
                json.dump(self.database, f, indent=4, ensure_ascii=False)
    '''
    # ------------------------------
    # INTERNAL UTILITIES
    # ------------------------------
    '''def _save_pdf_to_db(self, file_path, arxiv_id=None):
        """Assigns a DB filename and copies the PDF inside /data/pdfs."""
        pdf_name = f"{arxiv_id}.pdf" if arxiv_id else f"{uuid.uuid4()}.pdf"
        out_path = os.path.join(self.pdf_folder, pdf_name)
        with open(file_path, "rb") as src, open(out_path, "wb") as dst:
            dst.write(src.read())
        return pdf_name

    def _create_database_entry(self, title, pdf_name, researcher):
        entry = {
            "id": str(uuid.uuid4()),
            "title": title,
            "pdf_name": pdf_name,
            "researcher": researcher,
            "chunks": []  # filled after processing
        }
        self.database.append(entry)
        self.save_database()
        return entry
    

    # ------------------------------
    # MAIN UPLOAD METHODS
    # ------------------------------

    def upload_pdf(self, file_path, title, researcher, arxiv_id):
        """UPLOAD from a local file path and index it immediately."""
        self.database = self.load_database()
        pdf_name = self._save_pdf_to_db(file_path, arxiv_id)
        entry = self._create_database_entry(title, pdf_name, researcher)
        self.process_pdf(entry)
        print(f"Uploaded & indexed: {title}")
        return entry'''
    
    # ------------------------------
    # METADATA
    # ------------------------------
    '''def build_metadata_string(self, entry): #entry is dict
        m = entry.get("llm_metadata") or {}
        parts = []
        if m.get("title"):
            parts.append(f"Title: {m['title']}")
        if m.get("authors"):
            parts.append("Authors: " + ", ".join(m["authors"][:10]))
        if m.get("year"):
            parts.append(f"Year: {m['year']}")
        if m.get("topics"):
            parts.append("Topics: " + ", ".join(m["topics"]))
        if m.get("keywords"):
            parts.append("Keywords: " + ", ".join(m["keywords"]))
        if m.get("one_sentence_summary"):
            parts.append("Summary: " + m["one_sentence_summary"])

        if not parts:
            return ""
        
        META_CACHE = DATA_DIR / "llm_metadata_cache.json"

        if META_CACHE.exists():
            with open(META_CACHE, "r", encoding="utf-8") as f:
                metadata_cache = json.load(f)
        else:
            metadata_cache = {}

        return "[METADATA]\n" + "\n".join(parts)


    def ensure_llm_metadata(self, entry, doc_text):
        """
        Adds entry['llm_metadata'] if missing. Calls Gemini once per document.
        """
        if entry.get("llm_metadata") is not None:
            return  # already present (even if empty dict)

        try:
            entry["llm_metadata"] = extract_llm_metadata(doc_text)
            self.save_database()  # persist early to avoid repeated calls
        except Exception as e:
            print("Gemini metadata extraction failed:", e)
            entry["llm_metadata"] = {}
            self.save_database()  '''    

    # ------------------------------
    # PROCESSING (CHUNK + EMBEDDING)
    # ------------------------------
    def extract_text(self, pdf_path):
        doc = fitz.open(pdf_path)
        text = "".join([page.get_text() for page in doc])
        doc.close()
        return text

    def chunk_text(self, text, max_chars=1000):
        return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]


    def process_pdf(self, pdf_path, mode = "baseline"):
        '''pdf_path = os.path.join(self.pdf_folder, entry["pdf_name"])
        if not os.path.exists(pdf_path):
            print("PDF missing:", pdf_path)
            return'''

        text = self.extract_text(pdf_path)
        chunks = self.chunk_text(text)

        llm_meta = {}
        meta_str = ""
        if mode in ("concat", "dual"):
            llm_meta = get_llm_metadata(pdf_path.name, text)
            meta_str = build_metadata_string(llm_meta)
    
        #metadata doc-level
        '''self.ensure_llm_metadata(entry, text)
        metadata_str = self.build_metadata_string(entry)
        '''
        
        chunk_records = []
        for chunk in chunks: 
            if mode == "baseline":
                emb = self.embedder.encode(chunk)
                chunk_records.append({"text": chunk, "embedding": emb})
            elif mode == "concat":
                emb = self.embedder.encode(chunk + "\n\n" + meta_str)
                chunk_records.append({"text": chunk, "embedding": emb})
            elif mode == "dual":
                emb_text = self.embedder.encode(chunk)
                emb_meta = self.embedder.encode(meta_str)
                chunk_records.append({"text": chunk, "embedding_text": emb_text, "embedding_meta": emb_meta})
        
        return {"pdf_name": pdf_path.name, "llm_metadata": llm_meta, "chunks": chunk_records}
        '''for chunk in chunks:
            if mode == "concat":
                text_to_embed = chunk + "\n\n" + metadata_str if metadata_str else chunk
                embedding = self.embedder.encode(text_to_embed)

            entry["chunks"].append({
                "id": str(uuid.uuid4()),
                "text": chunk,
                "embedding": embedding
            })

        self.save_database()
        print(f"Indexed {len(chunks)} chunks for: {entry['title']}")
        return entry'''

    def build_index(self, pdf_paths, mode = "baseline"):
        self.database = []
        for p in pdf_paths:
            entry = self.process_pdf(p, mode=mode)
            print("heeey")
            self.database.append(entry)
        return self.database

In [6]:
#baseline retriever

class BaselineRetriever:
    def __init__(self, database = None, database_file=DB_PATH, embedder = None):
        """
        database: list (in-memory database). If provided, we won't load from file.
        database_file: path to JSON, used if database is None.
        embedder: optional shared Embedder instance.
        """
        self.embedder = embedder or Embedder()
        self.database_file = database_file
        self.database = database #can be None or list

    def load_database(self):
        if self.database is not None:
            return self.database
        if os.path.exists(self.database_file):
            with open(self.database_file, "r", encoding="utf-8") as f:
                return json.load(f)
        return []

    def cosine_similarity(self, v1, v2):
        v1 = np.array(v1)
        v2 = np.array(v2)
        return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

    def search(self, query, threshold=0.70):
        """Returns papers ranked by similarity. threshold ~ 0.65-0.80 recommended"""    
        self.database = self.load_database()

        if len(self.database) == 0:
            print("Database is empty: no entries to search.\n")

        query_emb = self.embedder.encode(query)
        results = []

        for entry in self.database:
            if "chunks" not in entry:
                continue  # not processed yet
            best_score = 0
            best_chunk = None

            for chunk in entry["chunks"]:
                score = self.cosine_similarity(query_emb, chunk["embedding"])
                if score > best_score:
                    best_score = score
                    best_chunk = chunk
            
            if best_score >= threshold:
                results.append({
                    "paper_id": entry["id"],
                    "title": entry["title"],
                    "researcher": entry["researcher"],
                    "pdf_name": entry.get("pdf_name", ""),
                    "score": round(best_score, 3),
                    "sample_text": best_chunk["text"][:300] if best_chunk else ""
                })
        # Sort best match → worst
        results.sort(key=lambda x: x["score"], reverse=True)
        return results

In [7]:
embedder = Embedder()

#dm = DataManager(embedder = embedder, max_chars = 1000)
dm = DataManager()

pdf_paths = list(PDF_DIR.glob("*.pdf"))

import random
random.seed(0)
subset_paths = random.sample(pdf_paths, k=3)

print("PDFs to index:", len(subset_paths))

PDFs to index: 3


In [12]:
baseline_db = dm.build_index(subset_paths, mode="baseline")

heeey
heeey
heeey


In [9]:
baseline_db[0]["llm_metadata"]

{}

In [14]:
concat_db = dm.build_index(subset_paths, mode="concat")

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
Please retry in 24.326683828s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerMod

In [9]:
concat_db[0]["llm_metadata"]

NameError: name 'concat_db' is not defined

In [ ]:
dual_db = dm.build_index(pdf_paths, mode="dual")

In [14]:
baseline_retriever = BaselineRetriever(database = baseline_db, embedder = embedder)
#concat_retriever   = BaselineRetriever(concat_db, embedder)
#dual_retriever     = BaselineRetriever(dual_db, embedder) 

In [30]:
baseline_retriever.search("GPT-2 fine-tuning", threshold=0.6)

KeyError: 'id'

In [28]:
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-